<h1><font color="orange"><b>2-additive Choquet</b></font></h1>

In [1]:
import numpy as np

Prvý syntetický model

In [2]:
np.random.seed(42)

n = 5000
x1 = np.random.uniform(0,1,n)
x2 = np.random.uniform(0,1,n)

z = -2 + 4*(0.3*x1+0.2*x2+0.5*np.minimum(x1,x2))
p = 1/(1+np.exp(-z))

target = np.random.binomial(1,p,n)

In [3]:
import pandas as pd

In [4]:
df = pd.DataFrame({"x1":x1,"x2":x2,"y":target,"probability":p})
df

,x1,x2,y,probability
0,0.374540,0.393636,0,0.380704
1,0.950714,0.473436,1,0.614550
2,0.731994,0.854547,1,0.736143
3,0.598658,0.340004,1,0.418339
4,0.156019,0.869650,0,0.308953
...,...,...,...,...
4995,0.897397,0.857656,1,0.814318
4996,0.119381,0.897509,0,0.289056
4997,0.327843,0.946708,0,0.451766
4998,0.815745,0.397488,0,0.522949


In [5]:
df['y'].value_counts()

,count
y,
0,2834
1,2166


In [6]:
X = df[['x1','x2']]
X

,x1,x2
0,0.374540,0.393636
1,0.950714,0.473436
2,0.731994,0.854547
3,0.598658,0.340004
4,0.156019,0.869650
...,...,...
4995,0.897397,0.857656
4996,0.119381,0.897509
4997,0.327843,0.946708
4998,0.815745,0.397488


In [7]:
new = X.copy()
clmns = list(new.columns)
for i in range(len(clmns)):
  for j in range(i+1,len(clmns)):
    atr1=clmns[i]
    atr2=clmns[j]
    novy = f"min({atr1},{atr2})"
    new[novy]=[min(x,y) for x,y in zip(new[atr1],new[atr2])]
for i in range(len(clmns)):
  for j in range(i+1,len(clmns)):
    atr1=clmns[i]
    atr2=clmns[j]
    novy = f"max({atr1},{atr2})"
    new[novy]=[max(x,y) for x,y in zip(new[atr1],new[atr2])]
new

,x1,x2,"min(x1,x2)","max(x1,x2)"
0,0.374540,0.393636,0.374540,0.393636
1,0.950714,0.473436,0.473436,0.950714
2,0.731994,0.854547,0.731994,0.854547
3,0.598658,0.340004,0.340004,0.598658
4,0.156019,0.869650,0.156019,0.869650
...,...,...,...,...
4995,0.897397,0.857656,0.857656,0.897397
4996,0.119381,0.897509,0.119381,0.897509
4997,0.327843,0.946708,0.327843,0.946708
4998,0.815745,0.397488,0.397488,0.815745


In [8]:
import cvxpy as cp

In [9]:
X = new.to_numpy()
y = target
print(X,y)

[[0.37454012 0.39363552 0.37454012 0.39363552]
 [0.95071431 0.47343566 0.47343566 0.95071431]
 [0.73199394 0.85454739 0.73199394 0.85454739]
 ...
 [0.32784284 0.94670792 0.32784284 0.94670792]
 [0.81574538 0.39748799 0.39748799 0.81574538]
 [0.59731238 0.2171404  0.2171404  0.59731238]] [0 1 1 ... 0 0 0]


In [10]:
from sklearn.model_selection import train_test_split
indices = np.arange(n)
train_idx, test_idx = train_test_split(indices,test_size=0.33,stratify=y,random_state=42)
X_train = X[train_idx]
X_test = X[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

In [11]:
i,j = X_train.shape
print(i,j)

koef = cp.Variable(j)
intrcpt = cp.Variable()

equation = intrcpt + X_train @ koef
logloss = cp.sum(cp.logistic(equation)-cp.multiply(y_train,equation))
constraints = [koef>=0]
problem = cp.Problem(cp.Minimize(logloss/i),constraints)
result = problem.solve()

print(intrcpt.value, koef.value, problem.value)

3350 4
-1.973949068884458 [0.69906171 0.61858934 2.09095195 0.4981153 ] 0.6152002585105664


In [12]:
gama = sum(koef.value)
print('Gamma: ',gama)

m1 = (koef.value[0]+koef.value[-1])/gama
m2 = (koef.value[1]+koef.value[-1])/gama
m12 = (koef.value[2]-koef.value[-1])/gama
print('Mobiove koeficienty: ',m1,m2,m12)

print('Súčet Mobiových koeficientov: ',m1+m2+m12)
if m1+m12>=0 and m2+m12>=0:
  print('Podmienka monotónnosti pre Mobiove koeficienty splnená.')
else:
  print('Podmienka monotónnosti pre Mobiove koeficienty neplatí.')

Gamma:  3.906718293503568
Mobiove koeficienty:  0.30644057625712523 0.2858421193187786 0.4077173044240961
Súčet Mobiových koeficientov:  1.0
Podmienka monotónnosti pre Mobiove koeficienty splnená.


In [13]:
from scipy.special import expit
y_pred_proba = expit(X_test @ koef.value + intrcpt.value)
y_pred = (y_pred_proba >= 0.5).astype(int)

In [14]:
from sklearn.metrics import mean_squared_error
rmse = np.sqrt(mean_squared_error(p[test_idx],y_pred_proba))
rmse

np.float64(0.020126854088359947)

In [15]:
from sklearn.metrics import log_loss
logloss_test = log_loss(y_test, y_pred_proba)
logloss_test

0.6151062139957182

Porovnanie s modelom s voľnými koeficientmi

In [16]:
new2 = new[['x1','x2','min(x1,x2)']]
new2

,x1,x2,"min(x1,x2)"
0,0.374540,0.393636,0.374540
1,0.950714,0.473436,0.473436
2,0.731994,0.854547,0.731994
3,0.598658,0.340004,0.340004
4,0.156019,0.869650,0.156019
...,...,...,...
4995,0.897397,0.857656,0.857656
4996,0.119381,0.897509,0.119381
4997,0.327843,0.946708,0.327843
4998,0.815745,0.397488,0.397488


In [17]:
Xmin = new2.to_numpy()
y = target
print(Xmin,y)

[[0.37454012 0.39363552 0.37454012]
 [0.95071431 0.47343566 0.47343566]
 [0.73199394 0.85454739 0.73199394]
 ...
 [0.32784284 0.94670792 0.32784284]
 [0.81574538 0.39748799 0.39748799]
 [0.59731238 0.2171404  0.2171404 ]] [0 1 1 ... 0 0 0]


In [18]:
Xmin_train = Xmin[train_idx]
Xmin_test = Xmin[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

In [19]:
i,j = Xmin_train.shape
print(i,j)

koefmin = cp.Variable(j)
intrcptmin = cp.Variable()
equationmin = intrcptmin + Xmin_train @ koefmin
loglossmin = cp.sum(cp.logistic(equationmin)-cp.multiply(y_train,equationmin))
problemmin = cp.Problem(cp.Minimize(loglossmin/i))
resultmin = problemmin.solve()

print(intrcptmin.value, koefmin.value, problemmin.value)

if koefmin.value[0]+koefmin.value[2]>=0 and koefmin.value[1]+koefmin.value[2]>=0:
  print('Podmienka monotónnosti pre koeficienty splnená.')
else:
  print('Podmienka monotónnosti pre koeficienty neplatí.')

3350 3
-1.9739487637270925 [1.19717634 1.11670467 1.59283667] 0.6152002585105678
Podmienka monotónnosti pre koeficienty splnená.


In [20]:
y_pred_proba_min = expit(Xmin_test @ koefmin.value + intrcptmin.value)
y_pred_min = (y_pred_proba_min >= 0.5).astype(int)

In [21]:
rmse_min = np.sqrt(mean_squared_error(p[test_idx],y_pred_proba_min))
rmse_min

np.float64(0.02012687772341323)

In [22]:
loglossmin_test = log_loss(y_test, y_pred_proba_min)
loglossmin_test

0.6151062168817033

In [23]:
if rmse<rmse_min:
  print('Vyhráva nový model s rozdielom ',abs(rmse-rmse_min))
else:
  print('Vyhráva model s voľnými koeficientmi s rozdielom ',abs(rmse-rmse_min))

if logloss_test<loglossmin_test:
  print('Vyhráva nový model s rozdielom ',abs(logloss_test-loglossmin_test))
else:
  print('Vyhráva model s voľnými koeficientmi ',abs(logloss_test-loglossmin_test))

Vyhráva nový model s rozdielom  2.3635053282566876e-08
Vyhráva nový model s rozdielom  2.8859851175511153e-09


Pokus s krížovou validáciou

In [24]:
from sklearn.model_selection import StratifiedKFold
cv = StratifiedKFold(n_splits=10, shuffle=True,random_state=42)

rmse_cv = []
logloss_cv = []

for train_idx, test_idx in cv.split(X,y):
  X_train = X[train_idx]
  X_test = X[test_idx]
  y_train = y[train_idx]
  y_test = y[test_idx]
  i,j = X_train.shape

  koef = cp.Variable(j)
  intrcpt = cp.Variable()

  equation = intrcpt + X_train @ koef
  logloss = cp.sum(cp.logistic(equation)-cp.multiply(y_train,equation))
  constraints = [koef>=0]
  problem = cp.Problem(cp.Minimize(logloss/i),constraints)
  result = problem.solve()
  y_pred_proba = expit(X_test @ koef.value + intrcpt.value)
  y_pred = (y_pred_proba >= 0.5).astype(int)
  rmse = np.sqrt(mean_squared_error(p[test_idx],y_pred_proba))
  rmse_cv.append(rmse)
  logloss_test = log_loss(y_test, y_pred_proba)
  logloss_cv.append(logloss_test)

finalrmse = np.mean(rmse_cv)
finallogloss = np.mean(logloss_cv)
print('Priemerná RMSE:',finalrmse)
print('Priemerný LogLoss:',finallogloss)

Priemerná RMSE: 0.018289955814564933
Priemerný LogLoss: 0.6157474964457592


In [25]:
rmsemin_cv = []
loglossmin_cv = []

for train_idx, test_idx in cv.split(Xmin,y):
  X_train = Xmin[train_idx]
  X_test = Xmin[test_idx]
  y_train = y[train_idx]
  y_test = y[test_idx]
  i,j = X_train.shape

  koef = cp.Variable(j)
  intrcpt = cp.Variable()

  equation = intrcpt + X_train @ koef
  logloss = cp.sum(cp.logistic(equation)-cp.multiply(y_train,equation))
  problem = cp.Problem(cp.Minimize(logloss/i))
  result = problem.solve()
  y_pred_proba = expit(X_test @ koef.value + intrcpt.value)
  y_pred = (y_pred_proba >= 0.5).astype(int)
  rmse = np.sqrt(mean_squared_error(p[test_idx],y_pred_proba))
  rmsemin_cv.append(rmse)
  logloss_test = log_loss(y_test, y_pred_proba)
  loglossmin_cv.append(logloss_test)

finalrmsemin = np.mean(rmsemin_cv)
finalloglossmin = np.mean(loglossmin_cv)
print('Priemerná RMSE:',finalrmsemin)
print('Priemerný LogLoss:',finalloglossmin)

/usr/local/lib/python3.13/dist-packages/cvxpy/problems/problem.py:1510: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/cvxpy/problems/problem.py:1510: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


Priemerná RMSE: 0.018289681837287
Priemerný LogLoss: 0.6157475290197908


In [26]:
if finalrmse<finalrmsemin:
  print('Vyhráva nový model s rozdielom ',abs(finalrmse-finalrmsemin))
else:
  print('Vyhráva model s voľnými koeficientmi s rozdielom ',abs(finalrmse-finalrmsemin))

if finallogloss<finalloglossmin:
  print('Vyhráva nový model s rozdielom ',abs(finallogloss-finalloglossmin))
else:
  print('Vyhráva model s voľnými koeficientmi s rozdielom',abs(finallogloss-finalloglossmin))

Vyhráva model s voľnými koeficientmi s rozdielom  2.73977277934212e-07
Vyhráva nový model s rozdielom  3.2574031583187946e-08


Druhý syntetický model

In [27]:
z2 = -2 + 4*(0.2*x1+0.3*x2+0.5*np.maximum(x1,x2))
p2 = 1/(1+np.exp(-z2))

target2 = np.random.binomial(1,p2,n)

In [28]:
df2 = pd.DataFrame({"x1":x1,"x2":x2,"y":target2,"probability":p2})
df2

,x1,x2,y,probability
0,0.374540,0.393636,0,0.391566
1,0.950714,0.473436,1,0.773840
2,0.731994,0.854547,1,0.789206
3,0.598658,0.340004,1,0.521050
4,0.156019,0.869650,1,0.712528
...,...,...,...,...
4995,0.897397,0.857656,0,0.823741
4996,0.119381,0.897509,1,0.724628
4997,0.327843,0.946708,1,0.784441
4998,0.815745,0.397488,1,0.681587


In [29]:
df2['y'].value_counts()

,count
y,
1,2787
0,2213


In [30]:
y2 = target2
y2

array([0, 1, 1, ..., 1, 1, 0], shape=(5000,))

In [31]:
train_idx, test_idx = train_test_split(indices,test_size=0.33,stratify=y2,random_state=42)
X_train2 = X[train_idx]
X_test2 = X[test_idx]
y_train2 = y2[train_idx]
y_test2 = y2[test_idx]

In [32]:
i,j = X_train2.shape
print(i,j)

koef2 = cp.Variable(j)
intrcpt2 = cp.Variable()

equation2 = intrcpt2 + X_train2 @ koef2
logloss2 = cp.sum(cp.logistic(equation2)-cp.multiply(y_train2,equation2))
constraints2 = [koef2>=0]
problem2 = cp.Problem(cp.Minimize(logloss2/i),constraints2)
result2 = problem2.solve()
print(intrcpt2.value, koef2.value, problem2.value)

3350 4
-1.9887209142947742 [0.26761027 0.92488082 0.56981614 2.22925492] 0.6118385169641317


In [33]:
gama2 = sum(koef2.value)
print('Gamma: ',gama2)
m1_2 = (koef2.value[0]+koef2.value[-1])/gama2
m2_2 = (koef2.value[1]+koef2.value[-1])/gama2
m12_2 = (koef2.value[2]-koef2.value[-1])/gama2
print('Mobiove koeficienty: ',m1_2,m2_2,m12_2)
print('Súčet Mobiových koeficientov',m1_2+m2_2+m12_2)

if m1_2+m12_2>=0 and m2_2+m12_2>=0:
  print('Podmienka monotónnosti pre Mobiove koeficienty splnená.')
else:
  print('Podmienka monotónnosti pre Mobiove koeficienty neplatí.')

Gamma:  3.991562142108901
Mobiove koeficienty:  0.6255358407665839 0.7902008356805093 -0.41573667644709317
Súčet Mobiových koeficientov 1.0
Podmienka monotónnosti pre Mobiove koeficienty splnená.


In [34]:
y_pred_proba2 = expit(X_test2 @ koef2.value + intrcpt2.value)
y_pred2 = (y_pred_proba2 >= 0.5).astype(int)

In [35]:
rmse2 = np.sqrt(mean_squared_error(p2[test_idx],y_pred_proba2))
rmse2

np.float64(0.018915938857301775)

In [36]:
logloss_test2 = log_loss(y_test2, y_pred_proba2)
logloss_test2

0.6156596175988449

Porovnanie s modelom s voľnými koeficientmi

In [37]:
Xmin_train2 = Xmin[train_idx]
Xmin_test2 = Xmin[test_idx]
y_train2 = y2[train_idx]
y_test2 = y2[test_idx]

In [38]:
i,j = Xmin_train2.shape
print(i,j)

koefmin2 = cp.Variable(j)
intrcptmin2 = cp.Variable()
equationmin2 = intrcptmin2 + Xmin_train2 @ koefmin2
loglossmin2 = cp.sum(cp.logistic(equationmin2)-cp.multiply(y_train2,equationmin2))
problemmin2 = cp.Problem(cp.Minimize(loglossmin2/i))
resultmin2 = problemmin2.solve()
print(intrcptmin2.value, koefmin2.value, problemmin2.value)

if koefmin2.value[0]+koefmin2.value[2]>=0 and koefmin2.value[1]+koefmin2.value[2]>=0:
  print('Podmienka monotónnosti pre koeficienty splnená.')
else:
  print('Podmienka monotónnosti pre koeficienty neplatí.')

3350 3
-1.9887212107775503 [ 2.49686622  3.15413593 -1.65944008] 0.6118385169641355
Podmienka monotónnosti pre koeficienty splnená.


In [39]:
y_pred_proba_min2 = expit(Xmin_test2 @ koefmin2.value + intrcptmin2.value)
y_pred_min2 = (y_pred_proba_min2 >= 0.5).astype(int)

In [40]:
rmsemin2 = np.sqrt(mean_squared_error(p2[test_idx],y_pred_proba_min2))
rmsemin2

np.float64(0.018915910002475708)

In [41]:
loglossmin_test2 = log_loss(y_test2, y_pred_proba_min2)
loglossmin_test2

0.615659615926816

In [42]:
if rmse2<rmsemin2:
  print('Vyhráva nový model s rozdielom ',abs(rmse2-rmsemin2))
else:
  print('Vyhráva model s voľnými koeficientmi s rozdielom ',abs(rmse2-rmsemin2))

if logloss_test2<loglossmin_test2:
  print('Vyhráva nový model s rozdielom ',abs(logloss_test2-loglossmin_test2))
else:
  print('Vyhráva model s voľnými koeficientmi ',abs(logloss_test2-loglossmin_test2))

Vyhráva model s voľnými koeficientmi s rozdielom  2.885482606670986e-08
Vyhráva model s voľnými koeficientmi  1.6720289597316196e-09


Pokus s krížovou validáciou

In [43]:
cv = StratifiedKFold(n_splits=10, shuffle=True,random_state=42)

rmse_cv2 = []
logloss_cv2 = []

for train_idx, test_idx in cv.split(X,y2):
  X_train = X[train_idx]
  X_test = X[test_idx]
  y_train = y2[train_idx]
  y_test = y2[test_idx]
  i,j = X_train.shape

  koef = cp.Variable(j)
  intrcpt = cp.Variable()

  equation = intrcpt + X_train @ koef
  logloss = cp.sum(cp.logistic(equation)-cp.multiply(y_train,equation))
  constraints = [koef>=0]
  problem = cp.Problem(cp.Minimize(logloss/i),constraints)
  result = problem.solve()
  y_pred_proba = expit(X_test @ koef.value + intrcpt.value)
  y_pred = (y_pred_proba >= 0.5).astype(int)
  rmse = np.sqrt(mean_squared_error(p2[test_idx],y_pred_proba))
  rmse_cv2.append(rmse)
  logloss_test = log_loss(y_test, y_pred_proba)
  logloss_cv2.append(logloss_test)

finalrmse2 = np.mean(rmse_cv2)
finallogloss2 = np.mean(logloss_cv2)
print('Priemerná RMSE:',finalrmse2)
print('Priemerný LogLoss:',finallogloss2)

Priemerná RMSE: 0.01878144552533923
Priemerný LogLoss: 0.6137899696543304


In [44]:
rmsemin_cv2 = []
loglossmin_cv2 = []

for train_idx, test_idx in cv.split(Xmin,y2):
  X_train = Xmin[train_idx]
  X_test = Xmin[test_idx]
  y_train = y2[train_idx]
  y_test = y2[test_idx]
  i,j = X_train.shape

  koef = cp.Variable(j)
  intrcpt = cp.Variable()

  equation = intrcpt + X_train @ koef
  logloss = cp.sum(cp.logistic(equation)-cp.multiply(y_train,equation))
  problem = cp.Problem(cp.Minimize(logloss/i))
  result = problem.solve()
  y_pred_proba = expit(X_test @ koef.value + intrcpt.value)
  y_pred = (y_pred_proba >= 0.5).astype(int)
  rmse = np.sqrt(mean_squared_error(p2[test_idx],y_pred_proba))
  rmsemin_cv2.append(rmse)
  logloss_test = log_loss(y_test, y_pred_proba)
  loglossmin_cv2.append(logloss_test)

finalrmsemin2 = np.mean(rmsemin_cv2)
finalloglossmin2 = np.mean(loglossmin_cv2)
print('Priemerná RMSE:',finalrmsemin2)
print('Priemerný LogLoss:',finalloglossmin2)

/usr/local/lib/python3.13/dist-packages/cvxpy/problems/problem.py:1510: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/cvxpy/problems/problem.py:1510: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/cvxpy/problems/problem.py:1510: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/cvxpy/problems/problem.py:1510: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/cvxpy/problems/problem.py:1510: UserWarn

Priemerná RMSE: 0.018781164707996022
Priemerný LogLoss: 0.6137900557792493


/usr/local/lib/python3.13/dist-packages/cvxpy/problems/problem.py:1510: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


In [45]:
if finalrmse2<finalrmsemin2:
  print('Vyhráva nový model s rozdielom ',abs(finalrmse2-finalrmsemin2))
else:
  print('Vyhráva model s voľnými koeficientmi s rozdielom ',abs(finalrmse2-finalrmsemin2))

if finallogloss2<finalloglossmin2:
  print('Vyhráva nový model s rozdielom ',abs(finallogloss2-finalloglossmin2))
else:
  print('Vyhráva model s voľnými koeficientmi s rozdielom',abs(finallogloss2-finalloglossmin2))

Vyhráva model s voľnými koeficientmi s rozdielom  2.8081734320881124e-07
Vyhráva nový model s rozdielom  8.6124918863284e-08
